In [1]:
import sys, os, gc
import warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multitest import multipletests
import pandas as pd

# --- 1. SILENCE WARNINGS ---
warnings.simplefilter(action='ignore')
pd.options.mode.chained_assignment = None

# --- CONFIG ---
start = Path.cwd()
REPO_ROOT = Path("/home/fsp585/Desktop/FiberphotometryCode/syllable_generation")
for p in (start, *start.parents):
    if p.name == "syllable_generation":
        REPO_ROOT = p
        break
DATA_DIR = REPO_ROOT / "analysis/replications/datta/processing/dlight_scalar_corr"

sns.set(style="whitegrid", context="talk", font_scale=0.8)
PLOT_HEIGHT = 3.0
PLOT_ASPECT = 1.1

# --- SEARCH SPACE (REMOVED L-R) ---
PLOT_CONFIG = {
    "pipelines":   ["datta_ransac", "standard_z"],
    "modes":       ["roll", "permute"],
    "colors":      ["G", "R"],          
    "hemispheres": ["L", "R"], # <--- Simplified
}

LABEL_MAP = {"velocity_px_s": "vel.", "angular_velocity": "ang. vel.", "acceleration": "acc.", 
             "jerk": "jerk", "transition": "trans.", "duration": "dur."}
FIXED_PALETTE = {"velocity_px_s": "C0", "transition": "C1", "angular_velocity": "C2", 
                 "acceleration": "C3", "jerk": "C4", "duration": "C5"}

In [2]:
# --- HELPERS ---
def load_newest(pattern):
    candidates = list(DATA_DIR.glob(f"{pattern}*.pkl"))
    if not candidates: raise FileNotFoundError(f"Missing: {pattern}")
    return pd.read_pickle(max(candidates, key=lambda p: p.stat().st_mtime))

def normalize_keys(df):
    for col in ["brain_reg", "neural_agg", "feature", "genotype", "hemisphere", "orig_letter"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
            if col == "hemisphere":
                df[col] = df[col].replace({"Left": "L", "Right": "R", "left": "L", "right": "R"})
            if col == "orig_letter":
                df[col] = df[col].replace({"Green": "G", "Red": "R"})
                
    if "timescale" in df.columns:
        df["timescale"] = pd.to_numeric(df["timescale"], errors="coerce").fillna(0).astype(int)
    return df

def apply_color_hotfix_stats(df):
    if df is None: return None
    names = df.index.names
    
    if "fiber_num" in names and "orig_letter" not in names:
        idx_df = df.index.to_frame()
        f_nums = pd.to_numeric(idx_df["fiber_num"], errors="coerce")
        idx_df["orig_letter"] = np.where(f_nums % 2 == 0, "G", "R")
        df.index = pd.MultiIndex.from_frame(idx_df).droplevel("fiber_num")
        
    if "hemisphere" in names:
        idx_df = df.index.to_frame()
        idx_df["hemisphere"] = idx_df["hemisphere"].replace({"Left": "L", "Right": "R", "left": "L", "right": "R"})
        df.index = pd.MultiIndex.from_frame(idx_df)
        
    return df

In [3]:
# --- PLOTTER ---
def plot_panel(geno, title_str, obs_g, chance_g, p_dist_g, features, output_path):
    if output_path.exists(): return

    valid_feats = [f for f in features if f in obs_g.columns]
    if not valid_feats: return

    # 1. P-Values
    final_p_long = pd.DataFrame()
    
    if p_dist_g is not None:
        p_vals = {}
        for ft in valid_feats:
            if ft not in p_dist_g.columns: continue
            
            # Mean across animals for Obs
            obs_mean = obs_g[ft].groupby(["timescale", "brain_reg", "neural_agg"]).mean().reset_index()
            # Mean across fibers for Stats
            shuff_dist = p_dist_g[ft].groupby(["timescale", "brain_reg", "neural_agg", "shuffle_index"]).mean().reset_index()
            
            obs_mean = normalize_keys(obs_mean)
            shuff_dist = normalize_keys(shuff_dist)
            
            merged = pd.merge(shuff_dist, obs_mean, on=["timescale", "brain_reg", "neural_agg"], suffixes=("_shuff", "_obs"))
            if merged.empty: continue

            merged["extreme"] = merged[f"{ft}_shuff"].abs() >= merged[f"{ft}_obs"].abs()
            p_vals[ft] = merged.groupby(["timescale", "brain_reg", "neural_agg"])["extreme"].mean()

        if p_vals:
            p_df = pd.DataFrame(p_vals)
            corrected = []
            if "brain_reg" in p_df.index.names:
                for reg, sub in p_df.groupby(level="brain_reg"):
                    if str(reg) == "UNK": continue
                    flat_p = sub.stack().values
                    if len(flat_p) == 0: continue
                    _, adj_p, _, _ = multipletests(flat_p, method="holm")
                    corrected.append(pd.Series(adj_p, index=sub.stack().index))
            if corrected:
                final_p_long = pd.concat(corrected).reset_index()
                final_p_long.columns = ["timescale", "brain_reg", "neural_agg", "variable", "p"]
                final_p_long = normalize_keys(final_p_long)

    # 2. Plotting Data
    reset_cols = ["timescale", "brain_reg", "neural_agg", "animal_id"]
    
    plot_data = obs_g[valid_feats].reset_index().melt(
        id_vars=[c for c in reset_cols if c in obs_g.index.names or c in obs_g.columns],
        var_name="variable", value_name="value"
    )
    plot_data = normalize_keys(plot_data)
    plot_data = plot_data[(plot_data["brain_reg"] != "UNK") & (plot_data["neural_agg"] != "UNK")]
    if plot_data.empty: return

    # 3. Draw
    # Standard CI=68 (SEM-ish) because we have animal_ids
    g = sns.relplot(
        data=plot_data, x="timescale", y="value", hue="variable",
        row="neural_agg", col="brain_reg", kind="line",
        errorbar=("ci", 68),
        height=PLOT_HEIGHT, aspect=PLOT_ASPECT, palette=FIXED_PALETTE, legend="auto"
    )
    
    if g.legend:
        g.legend.set_title("")
        for t in g.legend.get_texts():
            if t.get_text() in LABEL_MAP: t.set_text(LABEL_MAP[t.get_text()])

    g.set_axis_labels("Timescale (s)", "Correlation (r)")
    g.set(xlim=(0, 60))
    g.set_titles(row_template="{row_name}", col_template="{col_name}")

    # 4. Decorations
    for ax in g.axes.flat:
        if ax is None: continue
        title = ax.get_title()
        if "|" not in title: continue
        r_name, c_name = [t.strip() for t in title.split("|")]
        
        pretty_r = LABEL_MAP.get(r_name, r_name)
        ax.set_title(f"{pretty_r} | {c_name}")
        ax.axhline(0, color="gray", linewidth=0.8, zorder=0)

        # Chance Band
        if chance_g is not None:
            try:
                idx = chance_g.index
                mask = (idx.get_level_values("brain_reg").astype(str).str.strip() == c_name) & \
                       (idx.get_level_values("neural_agg").astype(str).str.strip() == r_name)
                panel_chance = chance_g[mask]
                envelope = panel_chance[panel_chance.index.get_level_values("feature").isin(valid_feats)]
                if not envelope.empty:
                    envelope = envelope.groupby("timescale").max()
                    envelope.index = pd.to_numeric(envelope.index)
                    ax.fill_between(envelope.index, -envelope.values, envelope.values, 
                                    color="#555555", alpha=0.25, linewidth=0, zorder=0.5)
            except Exception: pass

        # P-Values
        if not final_p_long.empty:
            p_sub = final_p_long[(final_p_long["brain_reg"] == c_name) & (final_p_long["neural_agg"] == r_name)]
            if not p_sub.empty:
                y_lims = ax.get_ylim()
                base_y = y_lims[1] - 0.05 * (y_lims[1]-y_lims[0])
                for i, ft in enumerate(valid_feats):
                    sig_ts = p_sub[(p_sub["variable"] == ft) & (p_sub["p"] < 0.05)]["timescale"]
                    if not sig_ts.empty:
                        ts = sig_ts.values
                        ax.hlines(y=[base_y - (i*0.06*(y_lims[1]-y_lims[0]))]*len(ts),
                                  xmin=ts-0.52, xmax=ts+0.52, colors=FIXED_PALETTE.get(ft, "k"),
                                  linewidth=3, zorder=10)

    g.fig.suptitle(title_str, y=1.02)
    g.fig.savefig(output_path, dpi=100, bbox_inches="tight")
    print(f"      [SAVED] {output_path.name}")
    sys.stdout.flush()
    plt.close(g.fig)

In [4]:
# ==============================================================================
# 5. MAIN
# ==============================================================================
if __name__ == "__main__":
    
    for pip in PLOT_CONFIG["pipelines"]:
        print(f"\n>>> PIPELINE: {pip}")
        
        try:
            obs_df = load_newest(f"datta_corr_OBSERVED_{pip}")
            if "r" in obs_df.columns:
                obs_df["r"] = pd.to_numeric(obs_df["r"], errors="coerce").astype("float32")
            
            # Hotfix
            if 'orig_letter' not in obs_df.columns and 'fiber_num' in obs_df.columns:
                f_nums = pd.to_numeric(obs_df['fiber_num'], errors='coerce')
                obs_df['orig_letter'] = np.where(f_nums % 2 == 0, 'G', 'R')
            
            obs_df = normalize_keys(obs_df)
            
            # PIVOT
            idx_cols = ["genotype", "timescale", "brain_reg", "neural_agg", "animal_id"]
            for c in ["hemisphere", "orig_letter"]:
                if c in obs_df.columns: idx_cols.append(c)
                
            obs_wide = obs_df.pivot_table(index=idx_cols, columns="feature", values="r")
            
        except FileNotFoundError:
            print("   Missing Observed.")
            continue

        for mode in PLOT_CONFIG["modes"]:
            print(f"   MODE: {mode}")
            try:
                chance_df, p_dist_df = load_newest(f"STATS_{pip}_{mode}")
                chance_df = apply_color_hotfix_stats(chance_df)
                p_dist_df = apply_color_hotfix_stats(p_dist_df)
            except FileNotFoundError:
                print("       [WARN] Missing stats.")
                chance_df, p_dist_df = None, None

            genotypes = obs_wide.index.get_level_values("genotype").unique()
            for geno in genotypes:
                if str(geno) == "UNK": continue
                
                for color in PLOT_CONFIG["colors"]:
                    for hemi in PLOT_CONFIG["hemispheres"]:
                        
                        try:
                            # --- SLICING (Simplified) ---
                            curr_obs = obs_wide.xs(geno, level="genotype")
                            
                            if "orig_letter" in curr_obs.index.names:
                                curr_obs = curr_obs.xs(color, level="orig_letter")

                            # --- HEMISPHERE FILTERING ---
                            if "hemisphere" in curr_obs.index.names:
                                curr_obs = curr_obs.xs(hemi, level="hemisphere")
                                
                                # Filter Stats as well
                                if chance_df is not None:
                                    try:
                                        curr_chance = chance_df.xs(geno, level="genotype").xs(color, level="orig_letter").xs(hemi, level="hemisphere")
                                    except KeyError: curr_chance = None
                                else: curr_chance = None
                                    
                                if p_dist_df is not None:
                                    try:
                                        curr_p = p_dist_df.xs(geno, level="genotype").xs(color, level="orig_letter").xs(hemi, level="hemisphere")
                                    except KeyError: curr_p = None
                                else: curr_p = None

                            else:
                                # Fallback if hemisphere is missing entirely
                                curr_chance, curr_p = None, None
                        
                        except KeyError: continue
                        
                        # --- PLOT ---
                        title = f"{geno} | {color}-{hemi} | {mode} | {pip}"
                        fname = f"scalar_{geno}_{color}_{hemi}_{mode}_{pip}.png"
                        
                        cols = curr_obs.columns
                        prim = [f for f in ["velocity_px_s", "transition"] if f in cols]
                        sec = [f for f in ["angular_velocity", "acceleration", "jerk", "duration"] if f in cols]
                        
                        if prim:
                            plot_panel(geno, title, curr_obs, curr_chance, curr_p, prim, DATA_DIR / f"PRIMARY_{fname}")
                        if sec:
                            plot_panel(geno, title, curr_obs, curr_chance, curr_p, sec, DATA_DIR / f"SECONDARY_{fname}")

    print("\nSearch Space Traversed.")


>>> PIPELINE: datta_ransac
   MODE: roll
      [SAVED] PRIMARY_scalar_KI_G_L_roll_datta_ransac.png
      [SAVED] SECONDARY_scalar_KI_G_L_roll_datta_ransac.png
      [SAVED] PRIMARY_scalar_KI_G_R_roll_datta_ransac.png
      [SAVED] SECONDARY_scalar_KI_G_R_roll_datta_ransac.png
      [SAVED] PRIMARY_scalar_KI_R_L_roll_datta_ransac.png
      [SAVED] SECONDARY_scalar_KI_R_L_roll_datta_ransac.png
      [SAVED] PRIMARY_scalar_KI_R_R_roll_datta_ransac.png
      [SAVED] SECONDARY_scalar_KI_R_R_roll_datta_ransac.png
      [SAVED] PRIMARY_scalar_WT_G_L_roll_datta_ransac.png
      [SAVED] SECONDARY_scalar_WT_G_L_roll_datta_ransac.png
      [SAVED] PRIMARY_scalar_WT_G_R_roll_datta_ransac.png
      [SAVED] SECONDARY_scalar_WT_G_R_roll_datta_ransac.png
      [SAVED] PRIMARY_scalar_WT_R_L_roll_datta_ransac.png
      [SAVED] SECONDARY_scalar_WT_R_L_roll_datta_ransac.png
      [SAVED] PRIMARY_scalar_WT_R_R_roll_datta_ransac.png
      [SAVED] SECONDARY_scalar_WT_R_R_roll_datta_ransac.png
   MODE: permu